# C++ Dataset Builder for SDV Error Classification

This notebook reads C++ files from the Vector Institute Images directory, joins them with `labeled_errors.csv`, extracts category metadata, and prepares clean reference code examples for later RAG-based classification.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/Users/fereshteh/Zebra_AI/HintGenerator')
CPP_ROOT = ROOT / 'AI Pilot/Vector_AI/SDV (C++)/Vector Institute Images'
CSV_PATH = ROOT / 'AI Pilot/Vector_AI/SDV (C++)/labeled_errors.csv'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('CPP_ROOT exists:', CPP_ROOT.exists())
print('CSV_PATH exists:', CSV_PATH.exists())
print('ARTIFACT_DIR:', ARTIFACT_DIR)

CPP_ROOT exists: True
CSV_PATH exists: True
ARTIFACT_DIR: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts


In [3]:
labels_df = pd.read_csv(CSV_PATH)
labels_df = labels_df.rename(columns={
    'file_name': 'filename',
    'error_description': 'error_description',
    'category': 'category'
})
labels_df['filename'] = labels_df['filename'].astype(str).str.strip()
labels_df['category'] = labels_df['category'].astype(str).str.strip().str.lower()
labels_df['error_description'] = labels_df['error_description'].astype(str).str.strip()

print('Rows in labeled_errors.csv:', len(labels_df))
labels_df.head(3)

Rows in labeled_errors.csv: 64


,filename,file_path,code_content,error_description,category
0,gyro_missing_resetYaw.cpp,/content/drive/MyDrive/Vector_AI/SDV (C++)/Vec...,#include <Arduino.h>\n#include <ZebraGyro.h>\n...,gyro missing resetYaw error,gyro error
1,gyro_reset_yaw_in_loop.cpp,/content/drive/MyDrive/Vector_AI/SDV (C++)/Vec...,#include <Arduino.h>\n#include <ZebraGyro.h>\n...,gyro reset yaw in loop error,gyro error
2,Gyro_read_without_begin.cpp,/content/drive/MyDrive/Vector_AI/SDV (C++)/Vec...,#include <Arduino.h>\n#include <ZebraGyro.h>\n...,Gyro read without begin error,gyro error


In [4]:
cpp_files = sorted(CPP_ROOT.rglob('*.cpp'))
cpp_rows = []
for p in cpp_files:
    try:
        code = p.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        code = ''
    cpp_rows.append({
        'filename': p.name,
        'abs_path': str(p),
        'relative_path': str(p.relative_to(ROOT)),
        'code_content': code,
    })

cpp_df = pd.DataFrame(cpp_rows)
print('Total .cpp files found:', len(cpp_df))
cpp_df.head(3)

Total .cpp files found: 115


,filename,abs_path,relative_path,code_content
0,Coding_always_true.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,#include <Arduino.h>\n\nint count = 0;\nvoid s...
1,Coding_reset_every_time.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,#include <Arduino.h>\n\nvoid setup() {}\nvoid ...
2,Inconsistance delay.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,"// write your code here\n#include ""Arduino.h""\..."


In [5]:
merged = cpp_df.merge(
    labels_df[['filename', 'error_description', 'category']],
    on='filename',
    how='left'
)

merged['has_error_label'] = merged['category'].notna()
merged['label'] = merged['category'].fillna('no error')
merged['error_description'] = merged['error_description'].fillna('No labeled error')

print('Labeled files:', int(merged['has_error_label'].sum()))
print('Unlabeled files:', int((~merged['has_error_label']).sum()))
merged[['filename', 'label', 'has_error_label']].head(10)

Labeled files: 64
Unlabeled files: 51


,filename,label,has_error_label
0,Coding_always_true.cpp,coding error,True
1,Coding_reset_every_time.cpp,coding error,True
2,Inconsistance delay.cpp,no error,False
3,Inconsistance delay_solution.cpp,no error,False
4,coding_always_true2.cpp,coding error,True
5,coding_case_sensitive.cpp,coding error,True
6,coding_empty_loop.cpp,coding error,True
7,coding_if_without_braces.cpp,coding error,True
8,coding_infinite_loop_inSetup.cpp,coding error,True
9,coding_joining_string_with_int.cpp,coding error,True


In [6]:
error_categories = sorted(labels_df['category'].dropna().unique().tolist())
all_categories_for_llm = ['no error'] + error_categories

categories_path = ARTIFACT_DIR / 'categories.json'
categories_path.write_text(
    json.dumps({
        'error_categories': error_categories,
        'all_categories_for_llm': all_categories_for_llm
    }, indent=2),
    encoding='utf-8'
)

print('Saved categories to:', categories_path)
print('Categories for model classification:')
for c in all_categories_for_llm:
    print('-', c)

Saved categories to: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts/categories.json
Categories for model classification:
- no error
- coding error
- colorsensor error
- dcmotor error
- distance sensor error
- gyro error
- screen error


In [7]:
library_dir = CPP_ROOT / 'Library'
clean_refs = []
if library_dir.exists():
    for p in sorted(library_dir.rglob('*.cpp')):
        code = p.read_text(encoding='utf-8', errors='ignore')
        clean_refs.append({
            'filename': p.name,
            'relative_path': str(p.relative_to(ROOT)),
            'label': 'no error',
            'error_description': 'Reference correct code from Library folder',
            'code_content': code
        })

clean_refs_path = ARTIFACT_DIR / 'clean_references.jsonl'
with clean_refs_path.open('w', encoding='utf-8') as f:
    for row in clean_refs:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Clean reference snippets saved:', len(clean_refs))
print('Path:', clean_refs_path)

Clean reference snippets saved: 5
Path: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts/clean_references.jsonl


In [8]:
dataset_out = merged[[
    'filename', 'abs_path', 'relative_path', 'code_content',
    'has_error_label', 'label', 'error_description'
]].copy()

dataset_csv = ARTIFACT_DIR / 'cpp_error_dataset.csv'
dataset_out.to_csv(dataset_csv, index=False)

print('Saved dataset:', dataset_csv)
dataset_out.sample(min(5, len(dataset_out)), random_state=42)

Saved dataset: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts/cpp_error_dataset.csv


,filename,abs_path,relative_path,code_content,has_error_label,label,error_description
81,TOF_reading_without_storing.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,#include <Arduino.h>\n#include <ZebraTOF.h>\n\...,True,distance sensor error,TOF reading without storing error
4,coding_always_true2.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,#include <Arduino.h>\n\nvoid setup() {}\nvoid ...,True,coding error,coding always true2 error
40,wrongcondition_solution.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,// Note: partial code shown — see full context...,False,no error,No labeled error
69,inconsistantMotorValues.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,"// write your code here\n#include ""Arduino.h""\...",False,no error,No labeled error
10,coding_loop_with_wrong_increment.cpp,/Users/fereshteh/Zebra_AI/HintGenerator/AI Pil...,AI Pilot/Vector_AI/SDV (C++)/Vector Institute ...,#include <Arduino.h>\n\nvoid setup() {\n Seri...,True,coding error,coding loop with wrong increment error


## Output Artifacts

This notebook creates:
- `artifacts/cpp_error_dataset.csv`
- `artifacts/categories.json`
- `artifacts/clean_references.jsonl`

Use these artifacts in the second notebook for OpenAI/Gemini classification and RAG comparison.